In [35]:
# Load PFF grades from CSV files
import pandas as pd

# Load QB grades from PFF QB.csv
pff_qb = pd.read_csv("PFF QB.csv")

# Team abbreviation mapping for QB data
PFF_QB_TO_STANDARD_ABBR = {
    'LA': 'LAR', 'ARZ': 'ARI', 'BLT': 'BAL', 
    'CLV': 'CLE', 'HST': 'HOU', 'IND': 'IND',
    'JAX': 'JAX', 'KC': 'KC', 'LV': 'LV',
    'DEN': 'DEN', 'LAC': 'LAC', 'NE': 'NE',
    'NYJ': 'NYJ', 'PIT': 'PIT', 'TEN': 'TEN',
    'CIN': 'CIN', 'BUF': 'BUF', 'ATL': 'ATL',
    'CAR': 'CAR', 'CHI': 'CHI', 'DAL': 'DAL',
    'DET': 'DET', 'GB': 'GB', 'MIN': 'MIN',
    'NO': 'NO', 'NYG': 'NYG', 'PHI': 'PHI',
    'SEA': 'SEA', 'SF': 'SF', 'TB': 'TB', 'WAS': 'WAS'
}

pff_qb['team'] = pff_qb['Team Abbreviation'].map(PFF_QB_TO_STANDARD_ABBR)

# Aggregate QB grades by team (average overall grade)
qb_grades = pff_qb.groupby('team', as_index=False)['Overall Grade'].mean()
qb_grades = qb_grades.rename(columns={'Overall Grade': 'qb'})

# Keep on 0-100 scale (Flask divides by 100)

print(f"QB grades loaded for {len(qb_grades)} teams")
qb_grades.head()

QB grades loaded for 31 teams


,team,qb
0,ARI,70.6
1,ATL,60.7
2,BAL,74.0
3,BUF,90.5
4,CAR,71.0


In [36]:
# Load Overall Grades CSV with team position grades
overall_grades = pd.read_csv("Overall Grades.csv")

# Map team names to standard NFL abbreviations
TEAM_NAME_TO_ABBR = {
    'Arizona Cardinals': 'ARI', 'Atlanta Falcons': 'ATL', 'Baltimore Ravens': 'BAL',
    'Buffalo Bills': 'BUF', 'Carolina Panthers': 'CAR', 'Chicago Bears': 'CHI',
    'Cincinnati Bengals': 'CIN', 'Cleveland Browns': 'CLE', 'Dallas Cowboys': 'DAL',
    'Denver Broncos': 'DEN', 'Detroit Lions': 'DET', 'Green Bay Packers': 'GB',
    'Houston Texans': 'HOU', 'Indianapolis Colts': 'IND', 'Jacksonville Jaguars': 'JAX',
    'Kansas City Chiefs': 'KC', 'Las Vegas Raiders': 'LV', 'Los Angeles Chargers': 'LAC',
    'Los Angeles Rams': 'LAR', 'Miami Dolphins': 'MIA', 'Minnesota Vikings': 'MIN',
    'New England Patriots': 'NE', 'New Orleans Saints': 'NO', 'New York Giants': 'NYG',
    'New York Jets': 'NYJ', 'Philadelphia Eagles': 'PHI', 'Pittsburgh Steelers': 'PIT',
    'San Francisco 49ers': 'SF', 'Seattle Seahawks': 'SEA', 'Tampa Bay Buccaneers': 'TB',
    'Tennessee Titans': 'TEN', 'Washington Commanders': 'WAS'
}

overall_grades['team'] = overall_grades['Team'].map(TEAM_NAME_TO_ABBR)

# Extract the relevant PFF grades (keep on 0-100 scale)
position_grades = overall_grades[[
    'team',
    'Running PFF Grade',        # For RB
    'Receiving PFF Grade',      # For WR/TE  
    'Run Blocking PFF Grade',   # For run blocking (reference)
    'Pass Blocking PFF Grade',  # For pass blocking (O-Line)
    'Defense PFF Grade'         # For DST
]].copy()

# Keep ALL grades on 0-100 scale (Flask divides by 100)
position_grades['rb'] = position_grades['Running PFF Grade']
position_grades['wrte'] = position_grades['Receiving PFF Grade']
position_grades['dst'] = position_grades['Defense PFF Grade']
position_grades['pass_blocking'] = position_grades['Pass Blocking PFF Grade']

# O-Line: Scale from actual range to 0-100 relative scale
min_pass_blocking = position_grades['pass_blocking'].min()
max_pass_blocking = position_grades['pass_blocking'].max()
position_grades['oline'] = 100 * (position_grades['pass_blocking'] - min_pass_blocking) / (max_pass_blocking - min_pass_blocking)

# Keep only needed columns
position_grades = position_grades[['team', 'oline', 'rb', 'wrte', 'dst']]

print(f"Position grades loaded for {len(position_grades)} teams")
print(f"O-Line scale: {position_grades['oline'].min():.2f} - {position_grades['oline'].max():.2f} (0-100 relative)")
print(f"RB scale: {position_grades['rb'].min():.2f} - {position_grades['rb'].max():.2f} (PFF 0-100)")
position_grades.head()

Position grades loaded for 32 teams
O-Line scale: 0.00 - 100.00 (0-100 relative)
RB scale: 68.10 - 91.40 (PFF 0-100)


,team,oline,rb,wrte,dst
0,ARI,36.860068,75.6,74.2,50.7
1,ATL,63.139932,84.8,74.1,65.7
2,BAL,44.709898,84.8,73.8,69.8
3,BUF,77.133106,90.4,74.9,60.8
4,CAR,60.409556,76.7,66.7,60.4


In [43]:
# Merge QB grades with position grades
AVgrades = pd.merge(position_grades, qb_grades, on='team', how='left')

# For teams without QB data (e.g., Miami), use league average
league_avg_qb = qb_grades['qb'].mean()
missing_qb_teams = AVgrades[AVgrades['qb'].isna()]['team'].tolist()
if missing_qb_teams:
    print(f"\n⚠ Teams without QB data: {missing_qb_teams}")
    print(f"  Using league average QB grade: {league_avg_qb:.2f}")
    AVgrades['qb'] = AVgrades['qb'].fillna(league_avg_qb)

# Add season column
AVgrades['season'] = 2025

# Create historical grades (2016-2024) using same values
historical_grades = []
for year in range(2016, 2025):
    year_data = AVgrades[['team', 'oline', 'qb', 'rb', 'wrte', 'dst']].copy()
    year_data['season'] = year
    historical_grades.append(year_data)

# Combine historical and current (2025 PFF) grades
AVgrades_all = pd.concat(historical_grades + [AVgrades], ignore_index=True)

print(f"\nFinal AVgrades: {len(AVgrades_all)} rows ({len(AVgrades)} teams × {len(range(2016, 2026))} seasons)")
print(f"Seasons: {sorted(AVgrades_all['season'].unique())}")
AVgrades_all.head(10)


⚠ Teams without QB data: ['MIA']
  Using league average QB grade: 73.91

Final AVgrades: 320 rows (32 teams × 10 seasons)
Seasons: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,team,oline,qb,rb,wrte,dst,season
0,ARI,36.860068,70.60,75.6,74.2,50.7,2016
1,ATL,63.139932,60.70,84.8,74.1,65.7,2016
2,BAL,44.709898,74.00,84.8,73.8,69.8,2016
3,BUF,77.133106,90.50,90.4,74.9,60.8,2016
4,CAR,60.409556,71.00,76.7,66.7,60.4,2016
5,CHI,79.863481,76.90,90.2,75.2,61.6,2016
6,CIN,38.907850,79.90,77.1,80.2,52.6,2016
7,CLE,0.000000,46.55,69.5,60.9,84.5,2016
8,DAL,13.651877,86.90,76.2,81.7,52.8,2016
9,DEN,100.000000,77.10,77.9,67.6,80.2,2016


In [44]:
# Select and organize final columns
AVgrades_final = AVgrades_all[['team', 'oline', 'qb', 'rb', 'wrte', 'dst', 'season']].copy()

print("Final AVgrades structure:")
print(AVgrades_final)
print(f"\nTotal rows: {len(AVgrades_final)}")
print(f"Total teams per season: {len(AVgrades_final[AVgrades_final['season']==2025])}")
print(f"Seasons: {sorted(AVgrades_final['season'].unique())}")
AVgrades_final.tail(10)

Final AVgrades structure:
    team      oline     qb    rb  wrte   dst  season
0    ARI  36.860068  70.60  75.6  74.2  50.7    2016
1    ATL  63.139932  60.70  84.8  74.1  65.7    2016
2    BAL  44.709898  74.00  84.8  73.8  69.8    2016
3    BUF  77.133106  90.50  90.4  74.9  60.8    2016
4    CAR  60.409556  71.00  76.7  66.7  60.4    2016
..   ...        ...    ...   ...   ...   ...     ...
315   SF  59.044369  81.05  71.8  80.9  49.4    2025
316  SEA  49.488055  79.00  91.4  89.2  84.4    2025
317   TB  66.894198  70.00  85.3  68.7  64.0    2025
318  TEN  66.552901  58.50  73.8  67.3  63.6    2025
319  WAS  75.085324  73.15  77.7  71.8  53.8    2025

[320 rows x 7 columns]

Total rows: 320
Total teams per season: 32
Seasons: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,team,oline,qb,rb,wrte,dst,season
310,NO,51.877133,72.30,68.8,72.4,71.0,2025
311,NYG,73.037543,68.40,84.3,65.5,59.7,2025
312,NYJ,63.481229,60.90,79.6,60.3,56.8,2025
313,PHI,83.276451,78.70,81.2,73.7,74.6,2025
314,PIT,84.641638,68.00,76.7,64.9,70.7,2025
315,SF,59.044369,81.05,71.8,80.9,49.4,2025
316,SEA,49.488055,79.00,91.4,89.2,84.4,2025
317,TB,66.894198,70.00,85.3,68.7,64.0,2025
318,TEN,66.552901,58.50,73.8,67.3,63.6,2025
319,WAS,75.085324,73.15,77.7,71.8,53.8,2025


In [45]:
# Save full grades with all columns (0-100 scale)
AVgrades_full = AVgrades_all[['team', 'oline', 'qb', 'rb', 'wrte', 'dst', 'season']].copy()
AVgrades_full.to_pickle("../PickleFiles/AVgrades.pkl")
print("Saved: AVgrades.pkl")
print(f"Shape: {AVgrades_full.shape}")
print(f"Grades are on 0-100 scale (Flask divides by 100)")

Saved: AVgrades.pkl
Shape: (320, 7)
Grades are on 0-100 scale (Flask divides by 100)


In [46]:
# Save position group grades (simplified version)
AVgrades_by_position = AVgrades_final[['team', 'oline', 'qb', 'rb', 'wrte', 'dst', 'season']].copy()
AVgrades_by_position.to_pickle("../PickleFiles/AVbyPositionGroup.pkl")
print("Saved: AVbyPositionGroup.pkl")
AVgrades_by_position

Saved: AVbyPositionGroup.pkl


,team,oline,qb,rb,wrte,dst,season
0,ARI,36.860068,70.60,75.6,74.2,50.7,2016
1,ATL,63.139932,60.70,84.8,74.1,65.7,2016
2,BAL,44.709898,74.00,84.8,73.8,69.8,2016
3,BUF,77.133106,90.50,90.4,74.9,60.8,2016
4,CAR,60.409556,71.00,76.7,66.7,60.4,2016
...,...,...,...,...,...,...,...
315,SF,59.044369,81.05,71.8,80.9,49.4,2025
316,SEA,49.488055,79.00,91.4,89.2,84.4,2025
317,TB,66.894198,70.00,85.3,68.7,64.0,2025
318,TEN,66.552901,58.50,73.8,67.3,63.6,2025


In [47]:
# Save current (2025) grades for Flask
currAVs = AVgrades_by_position[AVgrades_by_position['season'] == 2025][['team', 'oline', 'qb', 'rb', 'wrte', 'dst']].copy()
currAVs.to_pickle("../PickleFiles/currAVs.pkl")
print("Saved: currAVs.pkl (2025 season only)")
print(f"Teams: {len(currAVs)}")
currAVs.head()

Saved: currAVs.pkl (2025 season only)
Teams: 32


,team,oline,qb,rb,wrte,dst
288,ARI,36.860068,70.6,75.6,74.2,50.7
289,ATL,63.139932,60.7,84.8,74.1,65.7
290,BAL,44.709898,74.0,84.8,73.8,69.8
291,BUF,77.133106,90.5,90.4,74.9,60.8
292,CAR,60.409556,71.0,76.7,66.7,60.4


In [48]:
# Validation report
print("=" * 60)
print("PFF GRADE MAPPING SUMMARY (0-100 SCALE)")
print("=" * 60)
print(f"QB Grade: PFF QB Overall Grade (0-100)")
print(f"RB Grade: Running PFF Grade (0-100)")
print(f"WR/TE Grade: Receiving PFF Grade (0-100)")
print(f"O-Line Grade: Pass Blocking scaled 0-100 relative")
print(f"Defense Grade: Defense PFF Grade (0-100)")
print(f"\n⚠️ All grades are on 0-100 scale (Flask divides by 100)")
print("=" * 60)
print(f"\nData validation:")
print(f"  Total rows: {len(AVgrades_by_position)}")
print(f"  Seasons: {sorted(AVgrades_by_position['season'].unique())}")
print(f"  Teams per season: {len(AVgrades_by_position[AVgrades_by_position['season']==2025])}")
print(f"  O-Line range: {AVgrades_by_position['oline'].min():.1f} - {AVgrades_by_position['oline'].max():.1f}")
print(f"  QB range: {AVgrades_by_position['qb'].min():.1f} - {AVgrades_by_position['qb'].max():.1f}")
print(f"  RB range: {AVgrades_by_position['rb'].min():.1f} - {AVgrades_by_position['rb'].max():.1f}")
print("\n2025 PFF Grades (sample):")
print(AVgrades_by_position[AVgrades_by_position['season']==2025][['team', 'qb', 'rb', 'wrte', 'oline', 'dst']].head(10))

PFF GRADE MAPPING SUMMARY (0-100 SCALE)
QB Grade: PFF QB Overall Grade (0-100)
RB Grade: Running PFF Grade (0-100)
WR/TE Grade: Receiving PFF Grade (0-100)
O-Line Grade: Pass Blocking scaled 0-100 relative
Defense Grade: Defense PFF Grade (0-100)

⚠️ All grades are on 0-100 scale (Flask divides by 100)

Data validation:
  Total rows: 320
  Seasons: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
  Teams per season: 32
  O-Line range: 0.0 - 100.0
  QB range: 46.5 - 91.9
  RB range: 68.1 - 91.4

2025 PFF Grades (sample):
    team     qb    rb  wrte       oline   dst
288  ARI  70.60  75.6  74.2   36.860068  50.7
289  ATL  60.70  84.8  74.1   63.139932  65.7
290  BAL  74.00  84.8  73.8   44.709898  69.8
291  BUF  90.50  90.4  74.9   77.133106  60.8
292  CAR  71.00  76.7  66.7   60.409556  60.4
293  CHI  76.90  90.2  75.2   79.863481  61.6
294  CIN  79.90  77.1  80.2   38.907850 

In [49]:
# Quick verification of scale
print("SCALE VERIFICATION (should all be 0-100):")
print(f"O-Line: {currAVs['oline'].min():.1f} - {currAVs['oline'].max():.1f}")
print(f"QB: {currAVs['qb'].min():.1f} - {currAVs['qb'].max():.1f}")
print(f"RB: {currAVs['rb'].min():.1f} - {currAVs['rb'].max():.1f}")
print(f"WR/TE: {currAVs['wrte'].min():.1f} - {currAVs['wrte'].max():.1f}")
print(f"DST: {currAVs['dst'].min():.1f} - {currAVs['dst'].max():.1f}")

SCALE VERIFICATION (should all be 0-100):
O-Line: 0.0 - 100.0
QB: 46.5 - 91.9
RB: 68.1 - 91.4
WR/TE: 60.3 - 91.1
DST: 49.1 - 86.8
